## Imports

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import ipywidgets as widgets
from ipywidgets import interact_manual

from shared.mat_reader import MatReader
from shared.constants import CLASS_NAMES, DEVICE
from shared.utils import robust_minmax
from architectures.binary_ce.dataset import BinaryCEDataset
from architectures.binary_ce.model import BinaryCEModel

## Instantiation

In [ ]:
model = BinaryCEModel()
model_state_dict = torch.load("/Users/james/GitHub/lampe/docs/results/model.pth",
                              map_location=torch.device(DEVICE),
                              weights_only=True)
model.load_state_dict(model_state_dict)
model.to(DEVICE)

# enable gradients for grad-cam
model.eval()
for param in model.parameters():
    param.requires_grad = True

target_layers = [model.model.layer4[-1]]
targets = [BinaryClassifierOutputTarget(1)]
cam = GradCAM(model=model.model, target_layers=target_layers) # model.model: inner ResNet50 model

In [ ]:
# mat_reader = MatReader("/Users/james/GitHub/lampe/lampe_dataset/3x3 bad SHG removed/")
mat_reader = MatReader("/Users/james/GitHub/lampe/lampe_dataset/Full images/")
dataset = BinaryCEDataset(mat_reader, eff_fov_indices=list(range(mat_reader.get_num_fovs())), train=False)

print(f"MatReader shape: {mat_reader.images.shape}")

unique_rows, counts = np.unique(mat_reader.class_labels, axis=0, return_counts=True)

total_count = 0
for unique_row, count in zip(unique_rows, counts):
    print(f"{CLASS_NAMES[unique_row]}: [{total_count} - {total_count + count - 1}] (total: {count})")
    total_count += count

print(f"Bin Count: {np.bincount(mat_reader.class_labels)}")


## Grad-CAM

In [ ]:
def grad_cam_inference(idx: int, mask_thresh: float):
    
    # == Grad-CAM ==
    vis_img = robust_minmax(mat_reader.images[idx].transpose([1, 2, 0]), p_min=5.0, p_max=99.5)
    # print(vis_img.shape)

    # we need dataset to apply same preprocessing during inference as in training
    image, _class_label, _patient_id = dataset[idx]
    raw_image = torch.as_tensor(image, device=DEVICE)
    
    grayscale_cam = cam(input_tensor=raw_image.unsqueeze(0), targets=targets) # unsqueeze to add 1-element batch dim
    gradcam_img = show_cam_on_image(vis_img, grayscale_cam[0, :], use_rgb=True) # grayscale_cam[0, :] - take 1st element
    
    logit = cam.outputs
    confidence = torch.sigmoid(logit.squeeze())
    # print(confidence.item(), (confidence > 0.5).item())


    # == Mask ==
    mask = grayscale_cam[0, :] > mask_thresh
    masked_img = np.zeros_like(vis_img)
    masked_img[mask] = vis_img[mask]
    
    # inv_mask_thresh = 0.05
    #inv_mask = grayscale_cam[0, :] < mask_thresh
    inv_mask = ~mask
    inv_masked_img = np.zeros_like(vis_img)
    inv_masked_img[inv_mask] = vis_img[inv_mask]

    
    # == Plot ==
    fig, ax = plt.subplots(1, 4, figsize=(18, 5))

    true_class = f"Image Idx: {idx}  |  True: {CLASS_NAMES[mat_reader.class_labels[idx]]}  |  Sigmoid: {confidence:.4f}"
    fig.suptitle(true_class)
    
    ax[0].imshow(vis_img)
    ax[0].set_title('RGB (robust_minmax)')
    ax[0].axis('off')
    
    ax[1].imshow(gradcam_img)
    ax[1].set_title('Grad-CAM')
    ax[1].axis('off')
    
    ax[2].imshow(masked_img)
    ax[2].set_title('Masked')
    ax[2].axis('off')
    
    ax[3].imshow(inv_masked_img)
    ax[3].set_title('Inv. Masked')
    ax[3].axis('off')
    
    fig.tight_layout()

interact_manual.opts['manual_name'] = "Run Inference"
interact_manual(grad_cam_inference,
                idx=widgets.IntSlider(min=0, max=total_count, step=1, value=110, description="Image Index"),
                mask_thresh=widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description='Mask Threshold')
)